# 01 · Primeros pasos con Spark: SparkSession y RDDs

**Objetivos**
- Crear una `SparkSession` y explorar la Spark UI.
- Entender RDDs, particiones, transformaciones y acciones.
- Comprobar la **evaluación perezosa** (*lazy evaluation*).
- Implementar el clásico **WordCount**.

> Duración estimada: 30 min · Requisito: haber ejecutado `scripts/verificar_instalacion.py`

## 1. Crear la SparkSession
La `SparkSession` es el punto de entrada a Spark. De ella se obtiene el `SparkContext` (`sc`), necesario para trabajar con RDD (The Apache Software Foundation [ASF], 2026a).

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Fuera de Docker -> local[*]. Dentro del docker-compose -> spark://spark-master:7077
MASTER = os.environ.get("SPARK_MASTER", "local[*]")

builder = (SparkSession.builder
           .appName("01_RDD")
           .master(MASTER)
           .config("spark.sql.shuffle.partitions", "8"))   # pocos datos -> pocas particiones
if os.environ.get("SPARK_DRIVER_HOST"):
    builder = builder.config("spark.driver.host", os.environ["SPARK_DRIVER_HOST"])

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

DATA = os.path.abspath("../data")   # carpeta de datos del repositorio
print("Spark", spark.version, "| master:", sc.master)
print("Spark UI:", sc.uiWebUrl)

> Se recomienda abrir la URL de la **Spark UI** (normalmente http://localhost:4040) en otra pestaña del navegador y mantenerla abierta durante el cuaderno.

## 2. Crear RDDs
Un RDD se puede crear a partir de una colección de Python (`parallelize`) o de un archivo (`textFile`).

In [ ]:
numeros = sc.parallelize(range(1, 21), numSlices=4)   # 20 números en 4 particiones
print("Tipo:", type(numeros))
print("Particiones:", numeros.getNumPartitions())
print("Contenido por partición:", numeros.glom().collect())

## 3. Transformaciones (perezosas) y acciones

| Transformaciones | Acciones |
|---|---|
| `map`, `filter`, `flatMap`, `distinct`, `reduceByKey` | `collect`, `count`, `take`, `reduce`, `first` |

In [ ]:
cuadrados = numeros.map(lambda x: x ** 2)          # transformación
pares     = cuadrados.filter(lambda x: x % 2 == 0)  # transformación
print(pares)   # Aún no se ha calculado nada: solo existe un plan de ejecución.

In [ ]:
# Las acciones disparan la ejecución
print("collect:", pares.collect())
print("count  :", pares.count())
print("take(3):", pares.take(3))
print("suma   :", pares.reduce(lambda a, b: a + b))

### 3.1 Comprobando la evaluación perezosa
La celda siguiente define una transformación "lenta". Su definición es instantánea; el tiempo de cómputo aparece solo cuando se invoca la acción (Zaharia et al., 2012).

In [ ]:
import time

def lenta(x):
    time.sleep(0.1)
    return x * 10

t = time.time()
rdd_lento = numeros.map(lenta)
print(f"Definir la transformación: {time.time() - t:.3f} s")

t = time.time()
rdd_lento.collect()
print(f"Ejecutar la acción collect(): {time.time() - t:.3f} s  (4 particiones en paralelo)")

### 3.2 Linaje (lineage)
Spark registra cómo se construyó cada RDD; si se pierde una partición, la reconstruye a partir de ese registro (Zaharia et al., 2012).

In [ ]:
print(pares.toDebugString().decode())

## 4. RDDs de pares clave-valor
Muchas operaciones de Big Data se basan en pares `(clave, valor)`.

In [ ]:
ventas = sc.parallelize([
    ("Bogotá", 120), ("Medellín", 80), ("Cali", 60),
    ("Bogotá", 200), ("Cali", 40), ("Medellín", 100), ("Bogotá", 50),
])

total_ciudad = ventas.reduceByKey(lambda a, b: a + b)
print("Total por ciudad :", total_ciudad.collect())
print("Ventas por ciudad:", ventas.countByKey())
print("Ordenado desc    :", total_ciudad.sortBy(lambda kv: kv[1], ascending=False).collect())

> **`reduceByKey` frente a `groupByKey`**: `reduceByKey` combina valores dentro de cada partición *antes* del shuffle, con lo cual se transfieren menos datos por la red. Por esta razón se prefiere siempre que sea posible (Chambers & Zaharia, 2018).

## 5. WordCount: el "Hola Mundo" de Big Data

In [ ]:
import re

texto = sc.textFile(f"{DATA}/texto_bigdata.txt")
print("Líneas:", texto.count())
print(texto.first())

In [ ]:
stop = {"de","la","el","en","y","a","los","las","que","un","una","por","con","para","se","es","del","al","o","como"}

conteo = (texto
          .flatMap(lambda l: re.findall(r"\w+", l.lower()))
          .filter(lambda p: p not in stop and len(p) > 2)
          .map(lambda p: (p, 1))
          .reduceByKey(lambda a, b: a + b)
          .sortBy(lambda kv: kv[1], ascending=False))

for palabra, n in conteo.take(10):
    print(f"{palabra:<18}{n}")

> En la Spark UI, pestaña **Jobs**, el último job muestra un **DAG** con dos *stages*, debido a que `reduceByKey` produce un *shuffle*.

## 6. Persistencia (cache)
Si un RDD se reutiliza varias veces, guardarlo en memoria evita recalcularlo.

In [ ]:
grande = sc.parallelize(range(5_000_000), 8).map(lambda x: (x % 1000, x))

t = time.time(); grande.count(); print(f"Sin cache, 1ª vez: {time.time()-t:.2f} s")
grande.cache()
t = time.time(); grande.count(); print(f"Con cache, 1ª vez (materializa): {time.time()-t:.2f} s")
t = time.time(); grande.count(); print(f"Con cache, 2ª vez: {time.time()-t:.2f} s")
grande.unpersist();

> Antes de `unpersist()`, el RDD almacenado en caché puede observarse en la pestaña **Storage** de la Spark UI.

## Ejercicios propuestos

1. Calcular, a partir de `numeros`, la suma de los múltiplos de 3.
2. Contar, a partir de `texto`, cuántas palabras **distintas** contiene el archivo.
3. Identificar la palabra más larga del texto.
4. Calcular, con el RDD `ventas`, el **promedio** de venta por ciudad (sugerencia: `mapValues` y `reduceByKey` con tuplas `(suma, conteo)`).

In [ ]:
# Ejercicio 1

In [ ]:
# Ejercicio 2

In [ ]:
# Ejercicio 3

In [ ]:
# Ejercicio 4

In [ ]:
# Liberación de recursos al finalizar
spark.stop()

## Referencias

Chambers, B., & Zaharia, M. (2018). *Spark: The definitive guide. Big data processing made simple*. O'Reilly Media.

The Apache Software Foundation. (2026a). *Spark overview* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/

Zaharia, M., Chowdhury, M., Das, T., Dave, A., Ma, J., McCauley, M., Franklin, M. J., Shenker, S., & Stoica, I. (2012). Resilient distributed datasets: A fault-tolerant abstraction for in-memory cluster computing. En *Proceedings of the 9th USENIX Symposium on Networked Systems Design and Implementation* (pp. 15–28). USENIX Association.